# AlphaDog — treino de pose canina (Kaggle)

Por que Kaggle e nao Colab: **30 h de GPU por semana** (o Colab gratis acaba),
e o botao **Save & Run All** roda o notebook **em background no servidor**.
Voce fecha o navegador, desliga o PC, e ele termina sozinho. Foi a
desconexao que matou as tentativas anteriores.

## Antes de rodar — 3 ajustes na barra lateral direita

1. **Input → Add Input → Datasets**: anexe o dataset com o `yolo.zip`.
   (Crie em kaggle.com/datasets → New Dataset → suba `yolo.zip`. O Kaggle
   descompacta sozinho. Sobe **uma vez** e fica salvo pra sempre.)
2. **Accelerator**: `GPU T4 x2` (ou P100).
3. **Internet**: `On` (precisa pra baixar o ultralytics e o checkpoint base).

## Como rodar de verdade

Nao rode celula por celula e fique olhando. Clique em
**Save Version → Save & Run All (Commit)**. Fecha a aba. Volta depois.
O resultado fica em **Output**.


## 1. Confere a GPU

Se nao aparecer GPU, pare e ajuste o Accelerator.


In [ ]:
import subprocess

out = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(out or 'SEM GPU — ajuste Accelerator na barra lateral e rode de novo.')
assert 'MiB' in out, 'Sem GPU. Settings -> Accelerator -> GPU T4 x2.'


## 2. Instala o ultralytics


In [ ]:
!pip install -q ultralytics


## 3. Acha o dataset e copia para area gravavel

`/kaggle/input` e **somente leitura**, e o ultralytics precisa escrever o
`labels.cache` junto dos labels. Sem esta copia o treino falha com erro de
permissao. Sao ~500 MB, leva 1-3 min.


In [ ]:
import shutil
from pathlib import Path

WORK = Path('/kaggle/working')
DATA = WORK / 'data' / 'yolo'

if not (DATA / 'dogs.yaml').exists():
    found = list(Path('/kaggle/input').rglob('dogs.yaml'))
    if not found:
        raise SystemExit(
            'dogs.yaml nao encontrado em /kaggle/input. '
            'Anexou o dataset em Input -> Add Input?'
        )
    src = found[0].parent
    print('dataset encontrado em:', src)
    print('copiando para area gravavel...')
    DATA.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, DATA, dirs_exist_ok=True)

print('imagens:', sum(1 for _ in (DATA / 'images').rglob('*.jpg')))
print('labels :', sum(1 for _ in (DATA / 'labels').rglob('*.txt')))


## 4. Corrige o `path:` do yaml

O yaml foi gerado apontando para a maquina do Ramos. Reescreve para o Kaggle.


In [ ]:
YAML = DATA / 'dogs.yaml'
lines = [
    f'path: {DATA}' if l.startswith('path:') else l
    for l in YAML.read_text().splitlines()
]
YAML.write_text('\n'.join(lines))
print(YAML.read_text())


## 5. Treino

Duas travas que faltavam antes:

- **`time=3.0`**: teto de 3 horas. O ultralytics para sozinho e salva o melhor
  peso. Garante que o job **termina**, em vez de estourar a cota no meio.
  Era o erro original: 100 epocas = 3-4 h de T4 sem teto nenhum.
- **`resume`**: se ja existe `last.pt`, retoma de onde parou em vez de
  recomecar do zero.

60 epocas e o **teto**; com `patience=10` normalmente para antes, quando parar
de melhorar. Fine-tune de um checkpoint pre-treinado numa classe so converge
bem antes das 100 epocas originais.


In [ ]:
from ultralytics import YOLO

RUNS = WORK / 'runs'
last = RUNS / 'dogpose' / 'weights' / 'last.pt'

if last.exists():
    print('retomando de', last)
    model = YOLO(str(last))
    model.train(resume=True)
else:
    model = YOLO('yolo11n-pose.pt')
    model.train(
        data=str(YAML),
        epochs=60,
        time=3.0,
        imgsz=640,
        batch=32,
        patience=10,
        seed=1337,
        project=str(RUNS),
        name='dogpose',
        exist_ok=True,
        # Treino acontece em sala, quintal, luz variada: augment de cor ajuda,
        # geometrico agressivo atrapalha a leitura de pose.
        fliplr=0.5,
        flipud=0.0,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=5.0,
        # Mosaic corta membros e gera keypoint sem contexto: bom para deteccao,
        # ruim para pose. Desligado nas ultimas 10 epocas.
        close_mosaic=10,
    )


## 6. Exporta o TFLite

TFLite INT8 e o que vai para o Android. A quantizacao traz o FPS ao alvo e e
tambem onde a acuracia cai — por isso o gate roda sobre ELE, nao sobre o `.pt`.


In [ ]:
best = RUNS / 'dogpose' / 'weights' / 'best.pt'
assert best.exists(), f'nao achei {best} — o treino terminou?'

trained = YOLO(str(best))
tflite = trained.export(format='tflite', int8=True, imgsz=640, data=str(YAML))
print('tflite:', tflite)

onnx = trained.export(format='onnx', imgsz=640, simplify=True)
print('onnx :', onnx)


## 7. Junta os artefatos na raiz do Output

O que estiver em `/kaggle/working` vira o **Output** do notebook, baixavel
depois. Copiar para a raiz evita cacar em subpasta.


In [ ]:
import shutil
from pathlib import Path

OUT = Path('/kaggle/working/entrega')
OUT.mkdir(exist_ok=True)

for p in Path('/kaggle/working/runs').rglob('*'):
    if p.suffix in {'.tflite', '.onnx', '.pt'} or p.name == 'results.csv':
        dest = OUT / p.name
        shutil.copy(p, dest)
        print(f'{dest.name:35s} {dest.stat().st_size/1e6:8.1f} MB')

# O dataset copiado nao precisa ir junto no Output (500 MB a toa).
shutil.rmtree('/kaggle/working/data', ignore_errors=True)
print('\npronto. Baixe o .tflite na aba Output.')


## 8. Depois de baixar

Renomeie o `.tflite` para `dogpose.tflite` e coloque em:

```
apps/mobile/assets/models/dogpose.tflite
```

E avise o Claude — ligar o modelo no app e a proxima etapa.
